# Week 9b.1: Deep Agents: Backends and Permissions

On Tuesday, the files our deep agents wrote lived in the graph state. The part of a deep agent that decides where files live, and whether commands can run, is its **backend**.

Coding agents such as Claude Code work on real files on a real disk, and they run the code they write. Giving an agent that ability raises three questions:

| Question | What `deepagents` gives us |
|---|---|
| Where do the agent's files live? | a backend: the state, the store, a folder on disk, or a sandbox |
| Which files may it read or change? | permission rules for the file tools |
| Where does the code it writes run? | your machine, or a sandbox |

We'll answer the first two here. For the third, section 5 runs the agent's code on our machine, and W09b.2 moves it into a sandbox. At each step, we'll ask what the agent can reach that it should not.

Documentation: https://docs.langchain.com/oss/python/deepagents/backends

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. A workspace

The agent will work in a folder called `workspace` next to this notebook. The cell below creates it with two things in it:

- `data/papers.json`, the paper cache from Week 7
- `secrets/api_key.txt`, a fake API key

The fake key stands in for the files a real project folder holds. Your own `.env` file, with your real API keys, is a few folders above this one.

It also creates `outside_workspace.txt` next to the notebook, outside the workspace, so we can test whether the agent can reach it.

In [ ]:
import json
import shutil
from collections import Counter
from pathlib import Path

WORKSPACE = Path("workspace")

shutil.rmtree(WORKSPACE, ignore_errors=True) # delete it if it exists, first

(WORKSPACE / "data").mkdir(parents=True) # create the directory
(WORKSPACE / "secrets").mkdir()

shutil.copy("papers.json", WORKSPACE / "data" / "papers.json")
(WORKSPACE / "secrets" / "api_key.txt").write_text("FAKE-KEY-0000 (a stand-in for a real API key)\n")
Path("outside_workspace.txt").write_text("This file is outside the workspace.\n")



In [ ]:
def files_on_disk():
    return sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file())


files_on_disk()

In [ ]:
print(WORKSPACE)

`ask` runs one request and prints every tool call, what the tool returned, and the agent's reply, so that we can see which calls were allowed.

In [ ]:
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent


def ask(agent, text, thread="demo"):
    config = {"configurable": {"thread_id": thread}, "recursion_limit": 60}
    out = agent.invoke({"messages": [HumanMessage(content=text)]}, config)
    
    for m in out["messages"]:
        if m.type == "ai":
            for call in m.tool_calls:
                print(f"call:   {call['name']}({json.dumps(call['args'])[:110]})")
        elif m.type == "tool":
            print(f"result: {m.text[:140]}".replace("\n", " | "))
    print(f"\nreply:  {out['messages'][-1].text}")
    return out

## 2. Memory across threads

On Tuesday, every file the lead wrote belonged to one thread. A deep agent can also keep files that every thread can see, in a LangGraph store, which is a database for data that outlasts a thread.

A `CompositeBackend` sends file paths to different places by their prefix.

- Paths under `/memories/` go to a `StoreBackend`, so those files are saved in the store and are visible from every thread.
- Every other path goes to the `StateBackend`, which is the default, so those files belong to the thread.

`memory=["/memories/AGENTS.md"]` tells the deep agent to read that file when it starts and put its contents in the system prompt. 
- The file holds standing instructions that the agent, or the user, can edit.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore


def memory_namespace(runtime):
    """The folder in the store that holds the memory files. Every thread shares this one."""
    return ("memories",)


deep_store = #TODO
memories = StoreBackend(namespace=memory_namespace)
backend = CompositeBackend(default=StateBackend(), routes={"/memories/": memories})

This cell sets up three things.

- **`memory_namespace`.** A store is one database shared by everything that uses it, so it files each item under a **namespace**, a tuple of strings that works like a folder path: `("memories",)` is the folder `memories`, and `("memories", "alice")` would be `memories/alice`. 
- A `StoreBackend` has to know which folder its files go in, so we give it a function that returns one. The result must be a tuple even when it holds one name, and the comma is what makes `("memories",)` a tuple; `("memories")` is just the string `"memories"`. 
    - We pass a function instead of the tuple because the backend calls it on every read and write with the current run, so an app could return a different folder for each user. Ours returns the same folder every time, so every thread shares it.
- **`StoreBackend(namespace=memory_namespace)`.** We do not give it a store. During a run it uses the store the agent was built with, which is the `store=deep_store` we pass to `create_deep_agent` in the next cell.
- **`routes`.** A dictionary from path prefix to backend. A file operation on a path that starts with a key goes to that key's backend, and every other path goes to `default`. 
    - It is a dictionary because an agent can have several routes, for example `/memories/` to the store and `/data/` to a folder on disk; when two prefixes match, the longer one wins.

In [ ]:

deep = create_deep_agent(
    model=model,
    system_prompt="You are a helpful assistant.",
    backend=backend,
    checkpointer=InMemorySaver(),
    #TODO add store and memory,
)

In [ ]:
deep_store.search(("memories",))

On the first thread, we ask the agent to create `/memories/AGENTS.md` with a preference and to save a draft note to `/scratch.md`. The request says "create" because the file does not exist yet. Asked only to "save my preference", the agent may try `edit_file` on the missing file and fail.

In [ ]:
out = ask(deep, "Create the file /memories/AGENTS.md with my preference: answer in at most three bullet points. "
                "Also save a draft note 'call the bank' to /scratch.md.", "thread-1")

In [ ]:
print("\nfiles in thread-1's state:", list(out.get("files", {})))

In [ ]:
print("files in the store:       ", [item.key for item in deep_store.search(("memories",))])

`/scratch.md` is in the thread's state, and `/memories/AGENTS.md` is in the store. Now let's start a second thread and ask what it can see.

In [ ]:
_ = ask(deep, "List the files you can see, including inside /memories/.", "thread-2")

Only the memory file is there. The draft note belonged to `thread-1`.

On a third thread, we ask a question without mentioning the preference. The agent reads `/memories/AGENTS.md` when it starts, so its answer should follow the preference anyway.

In [ ]:
_ = ask(deep, "Why do LLM agents need long-term memory?", "thread-3")

### Inside the store

The store keeps each item under a **namespace**, a tuple of strings that works like a folder name. Each time the `StoreBackend` reads or writes a file, it calls `memory_namespace` to find out which folder to use. Ours always returns `("memories",)`, so every thread reads and writes the same files.

Let's look at what the store holds after the three threads.

In [ ]:
for item in deep_store.search(("memories",)):
    print("namespace:", item.namespace)
    print("key:      ", item.key)
    print("content:  ", item.value["content"])

The key is `/AGENTS.md`, not `/memories/AGENTS.md`. 

The `CompositeBackend` used the prefix to choose the backend and passed the rest of the path to it.

`namespace` takes a function, not a fixed tuple, because the function receives the `runtime` of the current run, and an app can use it to give each user a folder of their own. Here we write to one store through two `StoreBackend`s, one for each user, by calling them directly the way the file tools do. No model is called.

In [ ]:
users_store = InMemoryStore()


def alice_namespace(runtime):
    return ("memories", "alice")


def bob_namespace(runtime):
    return ("memories", "bob")


StoreBackend(store=users_store, namespace=alice_namespace).write("/AGENTS.md", "Answer in Spanish.")
StoreBackend(store=users_store, namespace=bob_namespace).write("/AGENTS.md", "Answer in at most three bullet points.")

In [ ]:
for item in users_store.search(("memories",)):
    print(item.namespace, item.key, "->", item.value["content"])

A namespace function that only returns a tuple is often written inline as a `lambda`, a one-line function without a name. 

`lambda runtime: ("memories", "carol")` 

- does the same as a `def` that takes `runtime` and returns that tuple. Here is a third user, written that way:

In [ ]:
StoreBackend(store=users_store, namespace=lambda runtime: ("memories", "carol")).write("/AGENTS.md", "Answer in one sentence.")

for item in users_store.search(("memories",)):
    print(item.namespace, item.key, "->", item.value["content"])

Both users have a file named `/AGENTS.md`, and each is in its own folder. 

In an app there would be one namespace function that reads the user's id from the `runtime`, so one agent could keep separate memories for every user.

## 3. Files on disk

`FilesystemBackend` keeps the agent's files in a folder on your disk. 

With `root_dir` set and `virtual_mode=True`, the agent's `/` is the workspace folder, so the agent's `/data/papers.json` is `workspace/data/papers.json` on your disk.

In [ ]:
from deepagents.backends import FilesystemBackend

disk = FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True)

`virtual_mode=True` 
- makes root_dir act as the root / of the agent's file system. 
- The agent can't name any path outside that folder.
- `virtual_mode=False` isn't safe as the agent could read files beyond the workspace (paths are taken literally and provide access to disk)

In [ ]:
agent = create_deep_agent(model=model, 
                          system_prompt="You work on the files in the workspace.",
                          backend=disk, 
                          checkpointer=InMemorySaver())

In [ ]:
_ = ask(agent, "List the files in the workspace, then write a two-line note to /notes/inventory.md that names each file.")

In [ ]:
files_on_disk()

The note is a real file now. You can open `workspace/notes/inventory.md` in your editor, and unlike the files in Tuesday's state, it is still there after the thread ends or the kernel restarts.

`virtual_mode=True` also keeps paths inside the root. A path that climbs out with `..` is refused before anything is read.

Let's call the backend directly, the same way the file tools call it.

In [ ]:
try:
    disk.read("/../outside_workspace.txt")
except ValueError as e:
    print("refused:", e)

Inside the root, though, nothing is off limits. The agent can read the key as easily as the paper cache.

In [ ]:
disk.read("/secrets/api_key.txt")

In [ ]:
disk.read("/secrets/api_key.txt").file_data["content"]

## 4. Permissions

`permissions` is a list of rules for the file tools. Each rule has:

- `operations`: `"read"`, `"write"`, or both
- `paths`: patterns such as `/secrets/**`, where `**` matches anything below that folder
- `mode`: `"allow"` (the default), `"deny"`, or `"interrupt"`, which pauses the run until a person approves the call

Each file tool counts as one of the two operations:

| operation | file tools |
|---|---|
| `"read"` | `ls`, `read_file`, `glob`, `grep` |
| `"write"` | `write_file`, `edit_file`, `delete` |

A rule that lists only `"write"` does not affect reading: the agent can still read the files it covers, but it cannot create, change, or delete them.

The rules are checked in order and the first rule that matches decides. A call that matches no rule is allowed.


In [ ]:
from deepagents import FilesystemPermission

rules = [
    FilesystemPermission(operations=["read", "write"], paths=["/secrets/**"], mode="deny"),
    FilesystemPermission(operations=["write"], paths=["/notes/**", "/report.md"]),
    FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),
]



Our rules say: 
- nothing under `/secrets/` may be read or written; 
- the agent may write under `/notes/` and to `/report.md`; every other write is denied. 
- Reads outside `/secrets/` match no rule, so they are allowed, which makes `/data/` read-only: the agent can read the papers but not change them. 

The order matters here. The rule that allows notes comes before the rule that denies all writes, so a write to `/notes/` is decided by the first one.

In [ ]:
agent = create_deep_agent(model=model,
                          system_prompt="You work on the files in the workspace. Do each step even if an earlier one fails.",
                          backend=disk, 
                          permissions=rules, 
                          checkpointer=InMemorySaver())


In [ ]:
_ = ask(agent, "Three steps: 1. read /secrets/api_key.txt. 2. write 'ok' to /config.json. 3. write 'ok' to /notes/check.md.")

In [ ]:
_ = ask(agent, "write data.txt to /data")

In [ ]:
_ = ask(agent, "read the first line from papers.json under /data")

In [ ]:
files_on_disk()

Each denied call came back to the model as an error message in its `ToolMessage`, and the agent reported what it could not do. Nothing was written to `/config.json`.

The rules apply to the agent's file tools. `FilesystemMiddleware` checks them each time a tool is called. Code that calls the backend directly, as we did in section 3, does not go through them.

## 5. Running code on your machine

To run code, a backend needs an `execute` method. 

`LocalShellBackend` is `FilesystemBackend` plus `execute`, which runs shell commands on your machine inside the workspace folder.

- `inherit_env=False` starts each command with an empty environment, so API keys in your environment variables are not passed to it. We give it a `PATH` that includes this notebook's Python, so that `python` works.
- `timeout=30` stops a command after 30 seconds.

**Important note about the path:**

The file tools use the agent's paths, such as `/data/papers.json`, but the shell uses real paths relative to the workspace folder, such as `data/papers.json`. The system prompt tells the agent this.

Let's ask a question that is easier to answer by running code than by reading: 
- how many cached papers are from each year.

In [ ]:
import os
import sys
from deepagents.backends import LocalShellBackend

shell = LocalShellBackend(root_dir=WORKSPACE, virtual_mode=True, inherit_env=False, timeout=30,
                          env={"PATH": os.path.dirname(sys.executable) + ":/usr/bin:/bin"})

In [ ]:
os.path.dirname(sys.executable)

In [ ]:
ANALYST_PROMPT = """You are a data analyst. Answer questions about the data by writing a Python script
and running it with execute.
The file tools see the workspace as /, so the data is /data/papers.json. Shell commands run inside the
workspace folder, so there the same file is data/papers.json."""

QUESTION = ("How many papers in /data/papers.json are from each year? The file has a 'papers' dictionary "
            "of records with a 'year' field. Write a script count_years.py, run it, and report the counts.")

In [ ]:
agent = create_deep_agent(model=model, 
                          system_prompt=ANALYST_PROMPT, backend=shell, 
                          checkpointer=InMemorySaver())


In [ ]:
_ = ask(agent, QUESTION)

Look at the `execute` calls. When the agent mixes up the two kinds of path, for example by running `python3 /count_years.py`, the shell's error comes back to it as the tool result, and it can correct the command on its next call.

Let's check the answer against a count we do ourselves.

In [ ]:
truth = Counter(p["year"] for p in json.load(open(WORKSPACE / "data" / "papers.json"))["papers"].values())
print(dict(sorted(truth.items())))

The shell does not know about `virtual_mode`. The file tools refuse a path outside the workspace, but a command can open any file your user account can open.

In [ ]:
try:
    shell.read("/../outside_workspace.txt")
except ValueError as e:
    print("read_file refused:", e)

print("execute ran:     ", shell.execute("cat ../outside_workspace.txt").output)

`../outside_workspace.txt` is harmless. A few more `../` reach the `.env` file with your own API keys. `inherit_env=False` kept the keys out of the environment, but the file is still on the disk, and the shell can read it.

The library knows that path rules cannot limit a shell. If we try to give permission rules to a backend that can execute commands, `create_deep_agent` refuses:

In [ ]:
try:
    create_deep_agent(model=model, backend=shell, permissions=rules)
except NotImplementedError as e:
    print(e)

For code that an agent writes, we need a boundary outside the process that runs the agent. That boundary is a sandbox, which we build in W09b.2.

## 6. Choosing a backend

| Backend | Where the files live | `execute` | What the agent can reach | Suits |
|---|---|---|---|---|
| `StateBackend` (the default) | the graph state, per thread | no | its own thread's files | most agents, including web apps |
| `StoreBackend` | the store, across threads | no | its namespace in the store | memory |
| `CompositeBackend` | different backends by path prefix | if its default backend has it | each route's backend | mixing, as with `/memories/` in section 2 |
| `FilesystemBackend` | a folder on disk | no | every file under `root_dir`, unless permissions say otherwise | local tools on files you are willing to share |
| `LocalShellBackend` | a folder on disk | on your machine | everything your user account can | your own experiments only |
| a sandbox | a container or a remote machine | inside the sandbox | what you put in the sandbox | running code the agent writes (W09b.2) |

## 7. Things to try

1. Move the last rule, which denies every write, to the top of `rules`, and run the three steps again. Which writes succeed now?

2. Add a rule that interrupts writes to `/report.md`: `FilesystemPermission(operations=["write"], paths=["/report.md"], mode="interrupt")`. Put it first, ask the agent to write a short report, and look at what `invoke` returns. We'll see how to answer an interrupt in a later session.

3. In section 2, ask the agent on a new thread to change the preference to "answer in one sentence". Then ask a question on yet another thread. Which preference does it follow?


In [ ]:
# your experiments here